In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import stim
import sinter
import matplotlib.pyplot as plt
import re
from bell_translation import translate_circuit, NoiseParams

%load_ext autoreload
%autoreload 2

In [ ]:
npoints = 8  # resolution for noise parameters
p_bell_z = 1e-3

p = np.logspace(-8, -6, npoints)

scan_ranges = [
    {
        "p_bell_x": p / 2,
        "p_bell_y": p / 2,
        "p_bell_z": np.full_like(p, p_bell_z),
        "p_internal": p,
        "p_m": p,
        "p_1": p / 10,
        "p_init": p,
    },
]


## Generate, Sample & Plot

In [ ]:
def parse_filename(name):
    patterns = {'n': r'n=(\d+)', 'k': r'k=(\d+)', 'd': r'd=(\d+)',
                'num_sub_rounds': r'num_sub_rounds=(\d+)', 'l': r'l=(\d+)'}
    params = {}
    for key, pat in patterns.items():
        m = re.search(pat, name)
        params[key] = int(m.group(1)) if m else None
    if params['num_sub_rounds']:
        params['rounds'] = (params['num_sub_rounds'] - 6) // 6
    return params


def build_tasks(circuit_files, scan_ranges):
    """Translate circuits in-memory for each noise point in scan_ranges."""
    tasks = []
    for circ_path in circuit_files:
        raw = stim.Circuit.from_file(str(circ_path))
        info = parse_filename(circ_path.name)
        for scan in scan_ranges:
            # Each scan dict maps param names to arrays of the same length
            n_points = len(next(iter(scan.values())))
            for i in range(n_points):
                noise = NoiseParams(**{k: float(v[i]) for k, v in scan.items()})
                translated = translate_circuit(raw, noise)
                # Use p_internal as the representative 'p' for x-axis
                metadata = {
                    'p': float(scan['p_internal'][i]),
                    'p_bell_z': float(scan['p_bell_z'][i]),
                    'n': info['n'],
                    'k': info['k'],
                    'expected_distance': info['d'],
                    'rounds': info.get('rounds'),
                    'l': info['l'],
                    'noise_params': noise.to_dict(),
                }
                tasks.append(sinter.Task(circuit=translated, json_metadata=metadata))
    return tasks

In [ ]:
# Sampling settings
max_shots = 100_000
max_errors = 100
num_workers = 10
decoder_name = 'pymatching'

# Resolve circuit files
circuit_dir = Path.cwd().parent / 'in/circuits/hyperbolic_floquet/8.8.8/k=10_d3a22e5e'
circuit_files = sorted(circuit_dir.glob('semi_hyperbolic_floquet_l=[123]_*.stim'))
print(f'{len(circuit_files)} circuits x {npoints} p-values x {len(scan_ranges)} scans = {len(circuit_files) * npoints * len(scan_ranges)} tasks')

tasks = build_tasks(circuit_files, scan_ranges)
print(f'{len(tasks)} tasks built. Sampling...')

results = sinter.collect(
    num_workers=num_workers,
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=[decoder_name],
    print_progress=True,
)

print(f'\nDone: {len(results)} results collected.')

3 circuits x 8 p-values x 1 scans = 24 tasks


Starting 4 workers...


24 tasks built. Sampling...


24 tasks left:
  workers    decoder eta shots_left errors_left json_metadata                                                                                                                                                                                                                                                                                                                                          
        1 pymatching <1m      98703          65 p=1e-08,bell_bias=100000.0,n=64,k=10,expected_distance=2,rounds=15,l=1,noise_params={'p_bell_x': 1e-08, 'p_bell_y': 1e-08, 'p_bell_z': 0.001, 'p_internal': 1e-08, 'p_m': 1e-08, 'p_1': 1e-09, 'p_init': 1e-08}                                                                                                                                        
        1 pymatching   ?     100000         100 p=1.9306977288832496e-08,bell_bias=100000.0,n=64,k=10,expected_distance=2,rounds=15,l=1,noise_params={'p_bell_x': 1.9306977288832496e-08, 'p_bell_y': 1.9


Done: 24 results collected.


In [ ]:
COLORS = {
    64: '#76b7b2',
    256: '#4e79a7',
    576: '#3B2B81',
}

def group_func(stat):
    m = stat.json_metadata
    pbz = m.get('p_bell_z', 0)
    return f'[[{m["n"]}, {m["k"]}, {m["expected_distance"]}]] p_bell_z={pbz:g}'

def plot_args_func(index, group_key, group_stats):
    m = group_stats[0].json_metadata
    color = COLORS.get(m['n'], '#999999')
    pbz = m.get('p_bell_z', 0)
    p = m.get('p', 0)
    ls = '-' if pbz <= p else '--'
    return {'color': color, 'linestyle': ls}

fig, ax = plt.subplots(1, 1, figsize=(6, 5))

sinter.plot_error_rate(
    ax=ax,
    stats=results,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    failure_values_func=lambda stat: stat.json_metadata['k'],
)

ax.loglog()
ax.set_xlabel('Physical Error Rate (p)')
ax.set_ylabel('Logical Error Rate per round')
ax.grid(which='major', linestyle='-', linewidth=0.5, alpha=0.7)
ax.grid(which='minor', linestyle=':', linewidth=0.3, alpha=0.5)

handles, labels = ax.get_legend_handles_labels()
def sort_key(label):
    n_match = re.search(r'\[\[(\d+)', label)
    pbz_match = re.search(r'p_bell_z=([0-9.eE+-]+)', label)
    return (int(n_match.group(1)), float(pbz_match.group(1)) if pbz_match else 0)
sorted_pairs = sorted(zip(handles, labels), key=lambda x: sort_key(x[1]))
if sorted_pairs:
    sorted_handles, sorted_labels = zip(*sorted_pairs)
    ax.legend(sorted_handles, sorted_labels, loc='lower right', frameon=True, edgecolor='black', fancybox=False)

fig.tight_layout()
fig.set_dpi(150)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

sinter.plot_error_rate(
    ax=ax,
    stats=results,
    x_func=lambda stat: stat.json_metadata.get('p_bell_z', 0),
    group_func=group_func,
    plot_args_func=plot_args_func,
    failure_values_func=lambda stat: stat.json_metadata['k'],
)

ax.loglog()
ax.set_xlabel('p_bell_z')
ax.set_ylabel('Logical Error Rate per round')
ax.grid(which='major', linestyle='-', linewidth=0.5, alpha=0.7)
ax.grid(which='minor', linestyle=':', linewidth=0.3, alpha=0.5)

handles, labels = ax.get_legend_handles_labels()
sorted_pairs = sorted(zip(handles, labels), key=lambda x: sort_key(x[1]))
if sorted_pairs:
    sorted_handles, sorted_labels = zip(*sorted_pairs)
    ax.legend(sorted_handles, sorted_labels, loc='lower right', frameon=True, edgecolor='black', fancybox=False)

fig.tight_layout()
fig.set_dpi(150)
plt.show()